# 01. Hybrid retrieval과 RRF

## 학습 목표

- lexical, semantic, recency 검색이 서로 다른 문서를 찾는 이유를 확인합니다.
- 서로 비교할 수 없는 score 대신 rank를 이용하는 RRF를 구현합니다.
- 여러 retriever의 합의가 단일 retriever의 강한 결과를 이길 수 있음을 관찰합니다.

외부 패키지 없이 실행되는 toy retrieval입니다. 실제 embedding model이나 Cerebras 내부 index를 재현하지 않습니다.

In [ ]:
from dataclasses import dataclass
from math import sqrt, exp
from datetime import datetime, timezone

@dataclass(frozen=True)
class Document:
    id: str
    text: str
    vector: tuple[float, ...]
    age_days: int
    rare_tokens: tuple[str, ...] = ()

docs = [
    Document("thread-new", "restore hangs after manifest load ERR_MANIFEST_TIMEOUT", (0.92, 0.18), 1, ("ERR_MANIFEST_TIMEOUT",)),
    Document("thread-semantic", "checkpoint stalls while reading the NFS manifest", (0.96, 0.10), 12, ("CKPT_PREFETCH",)),
    Document("runbook", "checkpoint restore runbook and NFS mount settings", (0.82, 0.30), 40),
    Document("chat-filler", "sounds good thanks I will try it", (0.78, 0.28), 0),
    Document("thread-old", "restore hangs after manifest load use legacy fetcher", (0.90, 0.12), 240),
]
query_text = "restore hangs after manifest load"
query_vector = (1.0, 0.0)

In [ ]:
def tokenize(text: str) -> set[str]:
    return {token.strip(".,:;!?()[]").lower() for token in text.split()}

def lexical_score(query: str, doc: Document) -> float:
    """정확히 겹치는 token 수를 사용합니다."""
    return len(tokenize(query) & tokenize(doc.text))

def cosine(a: tuple[float, ...], b: tuple[float, ...]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm = sqrt(sum(x*x for x in a)) * sqrt(sum(y*y for y in b))
    return dot / norm

def semantic_score(query_vector: tuple[float, ...], doc: Document) -> float:
    return cosine(query_vector, doc.vector)

def freshness_score(doc: Document, half_life_days: float = 90.0) -> float:
    """시간이 지나면 점수를 지수적으로 낮춥니다."""
    return exp(-doc.age_days / half_life_days)

def rare_token_score(doc: Document) -> float:
    return float(len(doc.rare_tokens))

scorers = {
    "lexical": lambda d: lexical_score(query_text, d),
    "semantic": lambda d: semantic_score(query_vector, d),
    "freshness": freshness_score,
    "rare-token": rare_token_score,
}

rankings = {
    name: [d.id for d in sorted(docs, key=scorer, reverse=True)]
    for name, scorer in scorers.items()
}
for name, ranking in rankings.items():
    print(f"{name:10s}: {ranking}")

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: dict[str, list[str]],
    k: float = 60.0,
    weights: dict[str, float] | None = None,
) -> list[tuple[str, float]]:
    """score(d) = sum_l weight_l / (k + rank_l(d))"""
    weights = weights or {}
    fused: dict[str, float] = {}
    for name, ranking in ranked_lists.items():
        weight = weights.get(name, 1.0)
        for rank, doc_id in enumerate(ranking, start=1):
            fused[doc_id] = fused.get(doc_id, 0.0) + weight / (k + rank)
    return sorted(fused.items(), key=lambda item: (-item[1], item[0]))

fused = reciprocal_rank_fusion(rankings)
for doc_id, score in fused:
    print(f"{doc_id:16s} {score:.6f}")

assert {doc_id for doc_id, _ in fused} == {d.id for d in docs}

## Weight와 smoothing constant 실험

오류 문자열이 중요한 운영 환경에서는 lexical과 rare-token 목록에 더 큰 weight를 줄 수 있습니다. 다만 production weight는 감으로 정하지 않고 label된 query-document 평가 세트에서 조정해야 합니다.

In [ ]:
operations_weights = {
    "lexical": 1.5,
    "semantic": 1.0,
    "freshness": 1.2,
    "rare-token": 1.5,
}

for k in (10, 60, 100):
    result = reciprocal_rank_fusion(rankings, k=k, weights=operations_weights)
    print(f"k={k:>3} -> {[doc_id for doc_id, _ in result]}")

print("\n확장 과제: 한 source가 여러 chunk로 top-K를 독점하지 않도록 source별 cap을 추가하세요.")